In [40]:
import torch
from torch import nn
import os
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

In [28]:
import splitfolders
splitfolders.ratio(input="data",output="dataset_split",seed=42,ratio=(.8,.0,.2))

Copying files: 49779 files [00:03, 15076.57 files/s]


In [54]:
from pathlib import Path
data_path = Path("")
image_path = data_path / "dataset_split"
train_path = image_path / "train"
test_path = image_path / "test"

In [34]:
def check_data(dir_path):
    for dirpath,dirnames,filenames in os.walk(dir_path):
        print(f"# of directories in '{len(dirnames)}' and {len(filenames)} images in {dirpath}")



In [36]:
check_data(image_path)

# of directories in '2' and 0 images in dataset_split
# of directories in '7' and 0 images in dataset_split/test
# of directories in '0' and 1184 images in dataset_split/test/surprise
# of directories in '0' and 1184 images in dataset_split/test/disgust
# of directories in '0' and 1184 images in dataset_split/test/angry
# of directories in '0' and 1307 images in dataset_split/test/sad
# of directories in '0' and 1184 images in dataset_split/test/fear
# of directories in '0' and 1634 images in dataset_split/test/neutral
# of directories in '0' and 2280 images in dataset_split/test/happy
# of directories in '7' and 0 images in dataset_split/train
# of directories in '0' and 4736 images in dataset_split/train/surprise
# of directories in '0' and 4736 images in dataset_split/train/disgust
# of directories in '0' and 4736 images in dataset_split/train/angry
# of directories in '0' and 5228 images in dataset_split/train/sad
# of directories in '0' and 4736 images in dataset_split/train/fear


In [55]:
NUM_WORKERS = os.cpu_count()

def create_dataloader(train_dir,
                      test_dir,
                      transforms: transforms.Compose,
                      batch_size:int,
                      workers:int = NUM_WORKERS):
    
    train_data = datasets.ImageFolder(root=train_dir,
                                      transform=transforms)
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)
    
    class_names = train_data.classes

    train_dataloader = DataLoader(dataset=train_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)
    
    test_dataloader = DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)
    return train_dataloader, test_dataloader, class_names

In [56]:
weight = models.EfficientNet_V2_S_Weights.DEFAULT

In [57]:
auto_transforms = weight.transforms()

In [58]:
auto_transforms.crop_size = [96]
auto_transforms.resize_size = [96]

In [59]:
auto_transforms

ImageClassification(
    crop_size=[96]
    resize_size=[96]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [60]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_path,
                                                                   test_dir=test_path,
                                                                   transforms=auto_transforms,
                                                                   batch_size=32,)

In [92]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [93]:
model = models.efficientnet_v2_s(weights=weight).to(device)

In [78]:
from torchinfo import summary
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        True
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 1280, 3, 3]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 24, 48, 48]          --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 24, 48, 48]          648                       True
│    │    └─BatchNorm2d: 3-2                            [32, 24, 48, 48]          [32, 24, 48, 48]          48                        True
│    │    └─SiLU: 3-3                                   [32, 24, 48, 48]          [32, 24, 48, 48]          --                        --
│    └─Sequential: 2-2  

In [79]:
for params in model.parameters():
    params.requires_grad = False

In [80]:
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        False
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 1280, 3, 3]          --                        False
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 24, 48, 48]          --                        False
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 24, 48, 48]          (648)                     False
│    │    └─BatchNorm2d: 3-2                            [32, 24, 48, 48]          [32, 24, 48, 48]          (48)                      False
│    │    └─SiLU: 3-3                                   [32, 24, 48, 48]          [32, 24, 48, 48]          --                        --
│    └─Sequential: 

In [81]:
output_shape = len(class_names)
output_shape

7

In [82]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [83]:
model.classifier = nn.Sequential(
    torch.nn.Linear(in_features=1280,out_features=output_shape)
)

In [84]:
for params in model.classifier.parameters():
    params.requires_grad = True

In [85]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [86]:
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 7]                   --                        Partial
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 1280, 3, 3]          --                        False
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 24, 48, 48]          --                        False
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 24, 48, 48]          (648)                     False
│    │    └─BatchNorm2d: 3-2                            [32, 24, 48, 48]          [32, 24, 48, 48]          (48)                      False
│    │    └─SiLU: 3-3                                   [32, 24, 48, 48]          [32, 24, 48, 48]          --                        --
│    └─Sequential

In [87]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(),lr = 0.001)

In [88]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device
              ):
    model.train() # Modelimizi train moduna alıyoruz.

    train_loss = 0 # Train Loss değerlerini tutmak için bir değişken oluşturuyoruz.
    train_acc = 0 # Train Accuracy değerlerini tutmak için bir değişken oluşturuyoruz.

    for batch, (X,y) in enumerate(dataloader): # Batch size gerekli değil burada.
        X,y = X.to(device), y.to(device)
        y_pred = model(X) # Modelimize bir tahminde bulunduruyoruz.

        loss = loss_fn(y_pred,y) # Loss değerlerimizi loss_fn ile hesaplıyoruz.
        train_loss += loss.item() # Çıkan loss değerlerini train_loss değişlenine toplayarak atıyoruz.

        # Modelimizi backpropagation yapıyoruz.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Softmax kullanarak modelimize label tahmininde bulunduruyoruz.
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

        train_acc += (y_pred_class == y).sum().item() / len(y_pred) # Accuracy değerlerimizi bir değişkende tutuyoruz.

    train_loss /= len(dataloader) # Train Loss değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz.
    train_acc /= len(dataloader) # Train Acc değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz
    return train_loss, train_acc # Geriye train_loss ve train_acc döndürüyoruz.

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
              device: torch.device
              ):
    model.eval() # Modelimizi test moduna alıyoruz.

    test_loss = 0 # test loss'ları tutmak için bir değişken oluşturuyoruz.
    test_acc = 0 # test accuracy'ları tutmak için bir değişken oluşturuyoruz.

    with torch.inference_mode(): # inference mode'a aldık.
        for batch, (X,y) in enumerate(dataloader): # batch gerekli değil fakat yine de aldık.
            X,y = X.to(device), y.to(device)
            test_pred = model(X) # modelimize tahmin ettiriyoruz.

            loss = loss_fn(test_pred,y) # loss'umuzu loss_fn ile hesaplıyoruz.
            test_loss += loss.item() # loss değerlerimizi test_loss değişkeninde topluyoruz.

            # Softmax activation function ile label tahmin ettiriyoruz.
            test_pred_label = torch.softmax(test_pred,dim=1).argmax(dim=1)

            acc = (test_pred_label == y).sum().item() / len(test_pred) # Calculate accuracy
            test_acc += acc # Accuracy değerlerimizi toplayıp test_acc değişkenine atıyoruz.

    test_loss /= len(dataloader) # Test loss değerlerimizi dataloader boyuna bölüyoruz.
    test_acc /= len(dataloader) # Test acc değerlerimizi dataloader boyuna bölüyoruz.

    return test_loss, test_acc # Geriye test_loss ve test_acc döndürüyoruz.

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               device:torch.device,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs:int = 10,
              ):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                           device = device
                                          )
        test_loss, test_acc = test_step(model = model,
                                           dataloader = test_dataloader,
                                           loss_fn = loss_fn,
                                           device = device
                                          )
        print(f"""
        Epoch:{epoch}
        Train Loss : {train_loss:.2f} -  Train Accuracy : {train_acc*100:.2f}
        Test Loss  : { test_loss:.2f} -  Test Accuracy  : {test_acc*100:.2f}
        """)
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)
    return results

In [95]:
results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=torch.device("cuda"),
                epochs=10)


        Epoch:0
        Train Loss : 8.17 -  Train Accuracy : 0.13
        Test Loss  : 785.47 -  Test Accuracy  : 0.11
        


KeyboardInterrupt: 